# 📊 **AP Statistics — Unit 1**
## 🟩 Lesson 3: Collecting Data & Sampling

> ### 💭 Investigation Question
> **How can we collect data from a sample so that we can make trustworthy conclusions about a population?**

Imagine our school has **2,000 students** and we want to estimate:

> **How many hours do students study each week?**

Surveying everyone may be difficult, so we decide to survey only **100 students**.

The central question is:

> ### **Which 100 students should we choose?**

## 🎯 Learning Goals

By the end of this investigation, you will be able to:

- Identify a **population, sample, population size, and sample size**.
- Explain the difference between a **sample** and a **census**.
- Distinguish between an **observational study** and an **experiment**.
- Explain why **random selection** matters.
- Distinguish sampling **with** and **without replacement**.
- Identify and simulate:
  - **Simple Random Sampling**
  - **Stratified Random Sampling**
  - **Cluster Random Sampling**
  - **Systematic Random Sampling**
- Explain **sampling variability**.
- Identify major sources of **sampling bias**.
- Decide whether results can reasonably be **generalized to a population**.

# 🏫 Meet the Population

For this lesson, we will use a **simulated school population of 2,000 students**.

Because this is a simulation, we know the true population values. This allows us to compare:

> **Population Value ↔ Sample Estimate**

The population is all 2,000 students:

$$
N=2000
$$

If we survey 100 students, then:

$$
n=100
$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
N = 2000

grades = rng.choice(
    ["Grade 9", "Grade 10", "Grade 11", "Grade 12"],
    size=N,
    p=[0.28, 0.26, 0.24, 0.22]
)

homerooms = [f"H{str(i).zfill(2)}" for i in range(1, 41)]
student_homerooms = rng.choice(homerooms, size=N)

grade_means = {
    "Grade 9": 5.0,
    "Grade 10": 6.0,
    "Grade 11": 7.0,
    "Grade 12": 8.0
}

study_hours = np.array([
    rng.normal(grade_means[g], 2.0) for g in grades
])
study_hours = np.clip(study_hours, 0, 20)

population = pd.DataFrame({
    "Student_ID": np.arange(1, N + 1),
    "Grade_Level": grades,
    "Homeroom": student_homerooms,
    "Weekly_Study_Hours": np.round(study_hours, 1)
})

display(population.head(10))
print("Population size:", len(population))

# 1️⃣ Census or Sample?

A **census** collects information from **every individual in the population**.

- Survey all 2,000 students → **Census**
- Survey 100 students → **Sample**

A sample is often used because a census may be too time-consuming, expensive, or difficult.

The challenge is to select a sample that represents the population well.

# 2️⃣ Observational Study vs Experiment

### Observational Study
Researchers **observe or measure** variables without assigning treatments.

Example:

> Ask students how many hours they study each week.

### Experiment
Researchers **assign treatments or conditions**.

Example:

> Randomly assign students to two study programs and compare test performance.

In this lesson, we focus mainly on **sampling and observational studies**.

Experimental design will be studied in Lesson 4.

# 3️⃣ Why Random Selection Matters

Suppose we survey the first 100 students who enter the library.

That may be convenient, but students who go to the library may study more than other students.

This could make the sample unrepresentative.

> ## **Random Selection**

Random selection uses chance to choose individuals from the population.

### Big Idea

> **Random Selection → supports generalization to the population**

Do not confuse this with:

> **Random Assignment → supports cause-and-effect conclusions**

# 4️⃣ Sampling With and Without Replacement

### Without Replacement
Once selected, an individual cannot be selected again.

### With Replacement
After selection, the individual is returned to the population and could be selected again.

In [ ]:
sample_without_replacement = population.sample(
    n=10, replace=False, random_state=1
)

sample_with_replacement = population.sample(
    n=10, replace=True, random_state=1
)

print("WITHOUT REPLACEMENT")
display(sample_without_replacement[["Student_ID", "Grade_Level"]])

print("\nWITH REPLACEMENT")
display(sample_with_replacement[["Student_ID", "Grade_Level"]])

# 5️⃣ Simple Random Sample (SRS)

A **Simple Random Sample** of size $n$ is selected so that **every possible sample of size $n$ has the same chance of being selected**.

Let's select 100 students from the full population.

In [ ]:
srs = population.sample(
    n=100,
    replace=False,
    random_state=10
)

display(srs.head())
print("SRS sample size:", len(srs))

## 📏 Population Mean vs Sample Mean

Because this population is simulated, we know the true population mean.

Let's compare it with the SRS estimate.

In [ ]:
population_mean = population["Weekly_Study_Hours"].mean()
srs_mean = srs["Weekly_Study_Hours"].mean()

print(f"Population mean: {population_mean:.2f} hours/week")
print(f"SRS sample mean: {srs_mean:.2f} hours/week")
print(f"Difference: {srs_mean - population_mean:+.2f} hours/week")

# 6️⃣ Stratified Random Sampling

Suppose grade level may affect study time.

We divide the population into **strata**:

- Grade 9
- Grade 10
- Grade 11
- Grade 12

Then we take a random sample from **every stratum**.

> ### **Stratified Sampling = SOME from ALL groups**

In [ ]:
stratified_sample = (
    population
    .groupby("Grade_Level", group_keys=False)
    .sample(n=25, random_state=20)
)

display(
    stratified_sample["Grade_Level"]
    .value_counts()
    .sort_index()
    .to_frame("Students Selected")
)

stratified_mean = stratified_sample["Weekly_Study_Hours"].mean()
print(f"Stratified sample mean: {stratified_mean:.2f}")

# 7️⃣ Cluster Random Sampling

Now suppose students naturally belong to homerooms.

A cluster sample might:

1. Randomly choose some homerooms.
2. Survey **every student** in those selected homerooms.

> ### **Cluster Sampling = ALL from SOME groups**

### Compare

- **Stratified:** some from every group
- **Cluster:** everyone from selected groups

In [ ]:
selected_clusters = (
    population["Homeroom"]
    .drop_duplicates()
    .sample(n=2, random_state=30)
    .tolist()
)

cluster_sample = population[
    population["Homeroom"].isin(selected_clusters)
]

cluster_mean = cluster_sample["Weekly_Study_Hours"].mean()

print("Selected homerooms:", selected_clusters)
print("Cluster sample size:", len(cluster_sample))
print(f"Cluster sample mean: {cluster_mean:.2f}")

# 8️⃣ Systematic Random Sampling

Suppose all 2,000 students are listed in order.

To obtain approximately 100 students:

$$
\frac{2000}{100}=20
$$

So after choosing a random starting point, we select every **20th student**.

Example:

`7, 27, 47, 67, ...`

In [ ]:
k = 20

rng_systematic = np.random.default_rng(50)
start = rng_systematic.integers(1, k + 1)

positions = np.arange(start, N + 1, k)
systematic_sample = population.iloc[positions - 1]

systematic_mean = systematic_sample["Weekly_Study_Hours"].mean()

print("Random starting point:", start)
print("Sampling interval:", k)
print("Sample size:", len(systematic_sample))
print(f"Systematic sample mean: {systematic_mean:.2f}")

# 9️⃣ Compare the Four Sampling Methods

| Method | Basic Idea |
|---|---|
| **Simple Random Sample** | Randomly select individuals from the full population |
| **Stratified Sample** | Sample some individuals from every stratum |
| **Cluster Sample** | Sample everyone from selected clusters |
| **Systematic Sample** | Random start, then select every $k$th individual |

### Memory Aid

> **Stratified = SOME from ALL**

> **Cluster = ALL from SOME**

In [ ]:
method_comparison = pd.DataFrame({
    "Method": [
        "Simple Random Sample",
        "Stratified Sample",
        "Cluster Sample",
        "Systematic Sample"
    ],
    "Sample Mean Study Hours": [
        srs_mean,
        stratified_mean,
        cluster_mean,
        systematic_mean
    ]
})

method_comparison["Difference from Population Mean"] = (
    method_comparison["Sample Mean Study Hours"] - population_mean
)

display(
    method_comparison.style
    .format({
        "Sample Mean Study Hours": "{:.2f}",
        "Difference from Population Mean": "{:+.2f}"
    })
    .hide(axis="index")
)

# 🔟 Sampling Variability

Different random samples contain different individuals.

Therefore, sample statistics naturally vary from sample to sample.

This is called **sampling variability**.

In [ ]:
sample_means = []

for seed in range(1, 11):
    temp_sample = population.sample(
        n=100,
        replace=False,
        random_state=seed
    )
    sample_means.append(
        temp_sample["Weekly_Study_Hours"].mean()
    )

sampling_variability_table = pd.DataFrame({
    "Sample Number": range(1, 11),
    "Sample Mean Study Hours": sample_means
})

display(
    sampling_variability_table.style
    .format({"Sample Mean Study Hours": "{:.2f}"})
    .hide(axis="index")
)

print(f"True population mean: {population_mean:.2f}")

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(
    sampling_variability_table["Sample Number"],
    sampling_variability_table["Sample Mean Study Hours"],
    marker="o"
)

plt.axhline(
    population_mean,
    linestyle="--",
    label="Population Mean"
)

plt.xlabel("Random Sample")
plt.ylabel("Sample Mean Study Hours")
plt.title("Sampling Variability Across 10 Random Samples")
plt.xticks(range(1, 11))
plt.legend()
plt.show()

# 1️⃣1️⃣ Sampling Bias

Bias is a **systematic problem** in the sampling or data-collection procedure.

## Voluntary Response Bias
Participants volunteer themselves.

## Undercoverage Bias
Part of the population is missing or less likely to be selected.

## Nonresponse Bias
Some selected individuals do not respond.

## Response Bias
Responses systematically differ from the true value because of things such as:
- leading wording,
- confusing questions,
- inaccurate self-reporting,
- social pressure.

# 🚫 Convenience Samples

A **convenience sample** chooses individuals who are easy to reach.

Example:

> Survey the first 100 students entering the cafeteria.

This may be quick, but it is not random and may not represent the population well.

> ### More data does not automatically fix a biased sampling method.

# 1️⃣2️⃣ Bias vs Sampling Variability

| Sampling Variability | Bias |
|---|---|
| Natural differences between random samples | Systematic error |
| Happens by chance | Comes from the method |
| Statistics vary around the population value | Statistics may be consistently too high or too low |
| Often reduced with larger random samples | Not necessarily fixed by a larger sample |

### Big Idea

A large **biased** sample can still produce a misleading result.

# 🌍 Can We Generalize?

At the end of a study, ask:

> **Who can our conclusion reasonably apply to?**

### Randomly Selected Sample
May support generalization to the population it was randomly selected from.

### Nonrandom Sample
Generalization to the full population may not be justified.

> ### **Random Selection helps determine WHO we can generalize to.**

# 🧪 Sampling Method Challenge

Identify the sampling method in each scenario.

### Scenario A
The school randomly selects 100 student ID numbers from the full student list.

### Scenario B
The school randomly selects 25 students from each grade.

### Scenario C
The school randomly selects three homerooms and surveys everyone in those homerooms.

### Scenario D
The school chooses a random starting student and surveys every 20th student.

### Scenario E
A teacher surveys the first 100 students entering the cafeteria.

For each scenario:

1. Identify the sampling method.
2. Decide whether it is random.
3. Identify any possible source of bias.
4. Decide whether results could reasonably be generalized.

# 🏁 Final Investigation — The Sampling Challenge

Your goal is to estimate:

> ### **The average weekly study time of all 2,000 students**

You may survey only about **100 students**.

Choose one method:

- Simple Random Sample
- Stratified Random Sample
- Cluster Random Sample
- Systematic Random Sample

Then:

1. Select your sample using Python.
2. Calculate the sample mean.
3. Compare it with the true population mean.
4. Calculate the difference.
5. Explain why your method is appropriate.
6. Identify possible weaknesses.
7. State whether you would generalize the result to the whole school.

---

## 🎯 Final Reflection

A good statistical conclusion depends on more than calculations.

Always ask:

> **Where did the data come from?**

> **Who was selected?**

> **How were they selected?**

> **Could the method introduce bias?**

> **Who can the results reasonably represent?**

# ✅ Lesson 3 Summary

You should now be able to explain:

- Population vs Sample
- Census
- Observational Study vs Experiment
- Random Selection
- Sampling With vs Without Replacement
- Simple Random Sampling
- Stratified Random Sampling
- Cluster Random Sampling
- Systematic Random Sampling
- Sampling Variability
- Voluntary Response Bias
- Undercoverage Bias
- Nonresponse Bias
- Response Bias
- Convenience Sampling
- Generalization

### Coming Next

# 🧪 Lesson 4 — Experimental Design

Sampling asks:

> **Who should we study?**

Experimental design asks:

> **How should treatments be assigned so that we can investigate cause and effect?**